# 주택담보대출 상환금 계산기

GitHub의 `mortgage_calculator.py`를 불러와 원리금균등, 원금균등, 만기일시 상환 방식을 비교합니다. 아래 셀을 위에서부터 순서대로 실행하세요.

In [ ]:
# 대출 설정값
PRINCIPAL = 100_000_000  # 대출 금액(원)
YEARS = 30               # 대출 기간(년)
ANNUAL_RATE = 4.5        # 연이율(%)

## 계산 모듈 불러오기

계산 로직의 원본은 GitHub의 `.py` 파일입니다. `SOURCE_REF`를 `main`으로 두면 항상 최신 버전을 사용하며, 특정 커밋 해시로 바꾸면 같은 버전을 재현할 수 있습니다.

In [ ]:
from importlib import reload
from urllib.request import urlretrieve
import pandas as pd
from IPython.display import display

SOURCE_REF = 'main'  # 재현이 필요하면 커밋 해시로 변경
MODULE_URL = (
    f'https://raw.githubusercontent.com/daedosee/lab/{SOURCE_REF}/'
    'mortgage_calculator/mortgage_calculator.py'
)
urlretrieve(MODULE_URL, 'mortgage_calculator.py')

import mortgage_calculator as mortgage_module
reload(mortgage_module)
MortgageCalculator = mortgage_module.MortgageCalculator
print(f'계산 모듈을 불러왔습니다: {MODULE_URL}')

In [ ]:
# 세 가지 상환 방식 계산 및 요약 비교
calculator = MortgageCalculator(PRINCIPAL, YEARS, ANNUAL_RATE)
results = [
    calculator.calculate_equal_principal_and_interest(),
    calculator.calculate_equal_principal(),
    calculator.calculate_bullet_maturity(),
]

summary = pd.DataFrame([{
    '상환 방식': result['method_name'],
    '첫 달 납입액': result['first_month_payment'],
    '마지막 달 납입액': result['last_month_payment'],
    '총 납부 이자': result['total_interest'],
    '총 상환 금액': result['total_payment'],
} for result in results])

summary_display = summary.copy()
for column in summary_display.columns[1:]:
    summary_display[column] = summary_display[column].map(
        lambda value: f'{value:,.0f}원'
    )
display(summary_display)

In [ ]:
# 월별 일정 확인 (0: 원리금균등, 1: 원금균등, 2: 만기일시)
SELECTED_METHOD = 0
schedule = pd.DataFrame(results[SELECTED_METHOD]['schedule']).rename(columns={
    'month': '회차',
    'monthly_payment': '월 납입액',
    'principal_payment': '원금',
    'interest_payment': '이자',
    'remaining_balance': '남은 원금',
})

schedule_display = schedule.copy()
for column in schedule_display.columns[1:]:
    schedule_display[column] = schedule_display[column].map(
        lambda value: f'{value:,.0f}원'
    )
display(schedule_display)

In [ ]:
# 선택한 상환 방식의 월별 일정을 CSV로 내려받기
CSV_FILENAME = 'mortgage_schedule.csv'
schedule.to_csv(CSV_FILENAME, index=False, encoding='utf-8-sig')
try:
    from google.colab import files
    files.download(CSV_FILENAME)
except ImportError:
    print(f'{CSV_FILENAME} 파일을 저장했습니다.')